In [1]:
import time
from typing import Literal

import torch

In [2]:
assert torch.cuda.is_available()

In [3]:
def bench(
    device: Literal["cpu", "cuda"],
    warmup: int,
    repeat: int,
    fn,
    *args: torch.Tensor,
):
    args = [arg.to(device) for arg in args]

    for _ in range(warmup):
        fn(*args)
    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for _ in range(repeat):
        fn(*args)
    if device == "cuda":
        torch.cuda.synchronize()

    t1 = time.perf_counter()

    elapsed = (t1 - t0) / repeat * 1000  # ms
    print(f"{elapsed:.3f} ms")

Matrix Multiplication over GF(2)

In [4]:
def matmul_GF2_direct(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return (x @ y) % 2

def matmul_GF2_indirect(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return (x.float() @ y.float()).round().int() % 2


In [5]:
num_iters = 10
batch_size = 1024
num_vars = 186
num_chks = 72

rng = torch.Generator()
rng.manual_seed(42)

ehat = torch.randint(0, 2, (num_iters, batch_size, num_vars), dtype=torch.int32, generator=rng)
chkmat = torch.randint(0, 2, (num_chks, num_vars), dtype=torch.int32, generator=rng)

In [6]:
# Check correctness
torch.equal(matmul_GF2_direct(ehat, chkmat.T), matmul_GF2_indirect(ehat, chkmat.T))

True

In [7]:
bench("cpu", 10, 50, matmul_GF2_direct, ehat, chkmat.T)
bench("cpu", 10, 50, matmul_GF2_indirect, ehat, chkmat.T)
bench("cpu", 10, 50, matmul_GF2_indirect, ehat.float(), chkmat.float().T)
try:
    bench("cuda", 10, 50, matmul_GF2_direct, ehat, chkmat.T)
except NotImplementedError as e:
    print("Not implemented on CUDA")
bench("cuda", 10, 50, matmul_GF2_indirect, ehat, chkmat.T)
bench("cuda", 10, 50, matmul_GF2_indirect, ehat.float(), chkmat.float().T)

12.591 ms
4.855 ms
3.876 ms
Not implemented on CUDA
0.070 ms
0.055 ms


Convert LLRs to ehat

In [8]:
num_iters = 10
batch_size = 1024
num_vars = 186
num_chks = 72

rng = torch.Generator()
rng.manual_seed(42)

llrs = torch.randn(num_iters, batch_size, num_vars)

In [9]:
def f1(x):
    return (x < 0).float()

def f2(x):
    return torch.where(x < 0, 1.0, 0.0)

In [10]:
# Check correctness
torch.equal(f1(llrs), f2(llrs))

True

In [11]:
bench("cpu", 10, 100, f1, llrs)
bench("cpu", 10, 100, f2, llrs)
bench("cuda", 10, 100, f1, llrs)
bench("cuda", 10, 100, f2, llrs)

1.183 ms
4.416 ms
0.017 ms
0.028 ms
